<a href="https://colab.research.google.com/github/hsy0828/fcnv2-weather-demo/blob/main/run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

GITHUB_USER = "hsy0828"
REPO_NAME = "fcnv2-weather-demo"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
TARGET_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(TARGET_DIR):
    !git clone {REPO_URL} {TARGET_DIR}
else:
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

print("=== 1. 安裝系統級庫 ===")
!apt-get update -qq && apt-get install -y -qq libeccodes-dev

print("=== 2. 安裝核心套件 ===")
!pip install --only-binary=:all: ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx || pip install ai-models ai-models-fourcastnetv2-gfs earthkit-meteo eccodes xarray cfgrib matplotlib cartopy onnxruntime onnx

print("=== 3. 降級鎖定 earthkit-data ===")
!pip install -q "earthkit-data==0.9.4" --force-reinstall --no-deps

print("=== 環境準備完成！請繼續執行 Cell 2 ===")

In [3]:
import os
import subprocess
import sys
import ai_models_fourcastnetv2_gfs
import ai_models_fourcastnetv2_gfs.model as fcnv2_module

%cd /content/fcnv2-weather-demo

# 1. 取得模型類別
target_model_cls = getattr(fcnv2_module, "FourCastNetv2", None)

print(f"成功載入模型類別：{target_model_cls.__name__}")

# 2. 設定預測參數 (6 ~ 72 小時)
lead_times = list(range(6, 78, 6))

print("=== 1. 開始執行 FourCastNetV2 氣象預測 ===")

try:
    # 加入 download_assets=True 以滿足 ai_models 父類別建構子需求
    model = target_model_cls(
        input="ecmwf-open-data",
        output="file",
        path="fourcastnetv2-small.grib",
        date=20240101,
        time=0,
        lead_time=lead_times,
        download_assets=True,
    )

    print("正在下載開放氣象資料與模型權重並進行推論...")
    model.run()
    print("🎉 預測成功！結果已儲存至 fourcastnetv2-small.grib")

    # 3. 自動繪圖
    print("\n=== 2. 開始執行自動繪圖 (plot_result.py) ===")
    if os.path.exists("plot_result.py"):
        subprocess.run(["python", "plot_result.py"], check=True)
        print("繪圖完成！")
    else:
        print("提示：專案目錄下未找到 plot_result.py，跳過繪圖步驟。")

except Exception as e:
    print(f"執行過程發生錯誤：{e}")
    import traceback
    traceback.print_exc()

ModuleNotFoundError: No module named 'ai_models_fourcastnetv2_gfs'